In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [9]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )



# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    'rafavzqz/geopdes',
    'Qiskit/openqasm',
    'mantidproject/mantid',
    'MIC-DKFZ/nnUNet',
    'CambridgeMolecularEngineering/tabledataextractor',
    'r-spatial/gstat',
    'LLNL/zfp',
    'ros-perception/slam_gmapping',
    'ofloveandhate/paramotopy',
    'qzhu2017/PyXtal'
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00096a8185cf670b800e75343f26413990d42303,rafavzqz_geopdes


In [ ]:
import time
import os
from tqdm import tqdm
import pandas as pd

# Keep project information attached to each SHA1
commit_list = df[['sha1', 'project']].copy()

# File used to save progress
progress_file = 'df_commit_data_progress.csv'
error_file = 'woc_errors.csv'

# Resume if previous progress exists
if os.path.exists(progress_file):
    df_existing = pd.read_csv(progress_file)
    completed_sha1s = set(df_existing['commit'].astype(str))
    commit_data = df_existing.to_dict('records')
    print(f"Resuming: {len(completed_sha1s)} commits already retrieved.")
else:
    completed_sha1s = set()
    commit_data = []
    print("Starting from the beginning.")

# Only retrieve commits that have not already been completed
remaining = commit_list[
    ~commit_list['sha1'].astype(str).isin(completed_sha1s)
].copy()

print(f"Remaining commits: {len(remaining)}")

# Maximum batch size is 10
chunks = [
    remaining.iloc[x:x+10]
    for x in range(0, len(remaining), 10)
]

commit_data = commit_data.copy()

for chunk_number, chunk in enumerate(tqdm(chunks), start=1):

    sha_list = chunk['sha1'].tolist()

    res, err = woc.get_values_many(
        'commit.tch',
        sha_list
    )

    res = {
        k: v[0]
        for k, v in res.items()
    }

    if err:
        print("Got Errors:", err)

        for sha1, error_message in err.items():
            # Don't add duplicates to the error list
            pass

    for commit_sha, commit in res.items():

        matching_projects = chunk.loc[
            chunk['sha1'] == commit_sha,
            'project'
        ]

        if len(matching_projects) == 0:
            continue

        project = matching_projects.iloc[0]

        commit_data.append({
            'project': project,
            'commit': commit_sha,
            'tree': commit[0],
            'parent': list(commit[1]),
            'author': commit[2][0],
            'author_time': int(commit[2][1]),
            'author_tz': commit[2][2],
            'committer': commit[3][0],
            'committer_time': int(commit[3][1]),
            'committer_tz': commit[3][2],
            'message': commit[4]
        })

    # Wait between requests
    time.sleep(1)

    # Save progress every 100 batches
    if chunk_number % 100 == 0:

        pd.DataFrame(commit_data).to_csv(
            progress_file,
            index=False
        )

        print(
            f"\nSaved progress: "
            f"{len(commit_data)} commits retrieved."
        )

# Final save
df_commit_data = pd.DataFrame(commit_data)

df_commit_data.to_csv(
    'df_commit_data.csv',
    index=False
)

print(f"\nFinished.")
print(f"Retrieved: {len(df_commit_data)} commits")


In [16]:
# check
# print("Total commits:", len(df))
# print("\nCommits by project:")
# print(df.groupby('project').size())
print("Number of SHA1s:", len(df))
print("Unique SHA1s:", df['sha1'].nunique())
print("Duplicate SHA1s:", df['sha1'].duplicated().sum())
print(df.head())


# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

Number of SHA1s: 190608
Unique SHA1s: 190608
Duplicate SHA1s: 0
                                       sha1           project
0  00096a8185cf670b800e75343f26413990d42303  rafavzqz_geopdes
1  00127306e06e39ecd0c0614a0337fe249f5d6041  rafavzqz_geopdes
2  0020ef4ab6245d64cafc3594452460aa043df21b  rafavzqz_geopdes
3  002e6cf594c43c3ad2b02e0c2bfa6a214b6bbdbb  rafavzqz_geopdes
4  003a1fd9a3f316f9446c64b6aa077488a3459781  rafavzqz_geopdes
{'project': 'rafavzqz_geopdes', 'commit': '00096a8185cf670b800e75343f26413990d42303', 'tree': 'bf0d44577592071b14183d93ea7dc8fe6090066a', 'parent': "['2e2002cce512677eba922b394083c443e67b4cb4']", 'author': 'rafavzqz <rafavzqz@de73d896-90e3-4c87-98df-8390668dc151>', 'author_time': 1309533879, 'author_tz': 0, 'committer': 'rafavzqz <rafavzqz@de73d896-90e3-4c87-98df-8390668dc151>', 'committer_time': 1309533879, 'committer_tz': 0, 'message': 'The examples in the article with the new version'}


In [19]:
dfinf = pd.DataFrame(commit_data)

dfinf = dfinf[
    ['project', 'commit', 'author', 'author_time', 'message']
].copy()

dfinf.columns = [
    'project',
    'commit',
    'author',
    'time',
    'message'
]

dfinf.head()

,project,commit,author,time,message
0,rafavzqz_geopdes,00096a8185cf670b800e75343f26413990d42303,rafavzqz <rafavzqz@de73d896-90e3-4c87-98df-839...,1309533879,The examples in the article with the new version
1,rafavzqz_geopdes,00127306e06e39ecd0c0614a0337fe249f5d6041,cdf <cdf@de73d896-90e3-4c87-98df-8390668dc151>,1294670579,stream function formulation finally works
2,rafavzqz_geopdes,0020ef4ab6245d64cafc3594452460aa043df21b,vazquez <vazquez@imati.cnr.it>,1617985169,Merge branch 'multipatch_C1' of github.com:raf...
3,rafavzqz_geopdes,002e6cf594c43c3ad2b02e0c2bfa6a214b6bbdbb,vazquez <vazquez@imati.cnr.it>,1447175722,Working on multipatch classes. The boundary is...
4,rafavzqz_geopdes,003a1fd9a3f316f9446c64b6aa077488a3459781,vazquez <vazquez@imati.cnr.it>,1447351824,Work on weak BC for RT\n


In [20]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,rafavzqz_geopdes,00096a8185cf670b800e75343f26413990d42303,rafavzqz <rafavzqz@de73d896-90e3-4c87-98df-839...,1309533879,The examples in the article with the new version


In [21]:
#mode a means append, so you have all your projects in the same file
yournetid='ltipto11'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

In [32]:
# Load the completed Part 1 CSV
dfinf = pd.read_csv(
    'ltipto11_project_summary.csv',
    sep=';',
    low_memory=False
)

# Convert timestamps to numbers
dfinf['time'] = pd.to_numeric(dfinf['time'], errors='coerce')

# Basic checks
print("Columns:")
print(dfinf.columns.tolist())

print("\nTotal commit records:")
print(len(dfinf))

print("\nCommits by project:")
print(dfinf['project_wocid'].value_counts())

print("\nMissing values:")
print(dfinf.isna().sum())

print("\nUnique commits:")
print(dfinf['commit_sha1'].nunique())

# Calculate required WoC statistics
woc_stats = dfinf.groupby('project_wocid').agg(
    number_of_commits=('commit_sha1', 'count'),
    number_of_authors=('author', 'nunique'),
    max_time=('time', 'max'),
    min_time=('time', 'min')
).reset_index()

print("\nWoC Statistics:")
print(woc_stats.to_string(index=False))


Columns:
['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']

Total commit records:
190363

Commits by project:
project_wocid
mantidproject_mantid                                176744
rafavzqz_geopdes                                      2582
mic-dkfz_nnunet                                       2575
qzhu2017_pyxtal                                       2317
llnl_zfp                                              2038
qiskit_openqasm                                       2033
ofloveandhate_paramotopy                               902
r-spatial_gstat                                        538
cambridgemolecularengineering_tabledataextractor       318
ros-perception_slam_gmapping                           316
Name: count, dtype: int64

Missing values:
project_wocid      0
commit_sha1        0
author             0
time               0
commit message    15
dtype: int64

Unique commits:
190363

WoC Statistics:
                                   project_wocid  number_of_commi

In [33]:
import requests
import pandas as pd

repos = {
    'rafavzqz_geopdes': 'rafavzqz/geopdes',
    'qiskit_openqasm': 'openqasm/openqasm',
    'mantidproject_mantid': 'mantidproject/mantid',
    'mic-dkfz_nnunet': 'MIC-DKFZ/nnUNet',
    'cambridgemolecularengineering_tabledataextractor': 'CambridgeMolecularEngineering/tabledataextractor',
    'r-spatial_gstat': 'r-spatial/gstat',
    'llnl_zfp': 'LLNL/zfp',
    'ros-perception_slam_gmapping': 'ros-perception/slam_gmapping',
    'ofloveandhate_paramotopy': 'ofloveandhate/paramotopy',
    'qzhu2017_pyxtal': 'MaterSim/PyXtal'
}

github_data = []

for project, repo in repos.items():
    # Get repository information
    repo_url = f'https://api.github.com/repos/{repo}'
    repo_response = requests.get(repo_url)
    repo_data = repo_response.json()

    # Get most recent commit
    commits_url = f'https://api.github.com/repos/{repo}/commits'
    commits_response = requests.get(commits_url)
    commits_data = commits_response.json()

    if commits_response.status_code == 200 and len(commits_data) > 0:
        last_commit_date = commits_data[0]['commit']['author']['date']
    else:
        last_commit_date = 'ERROR'

    github_data.append({
        'project_wocid': project,
        'stars': repo_data.get('stargazers_count'),
        'forks': repo_data.get('forks_count'),
        'last_commit_date': last_commit_date
    })

github_stats = pd.DataFrame(github_data)

github_stats


,project_wocid,stars,forks,last_commit_date
0,rafavzqz_geopdes,124,51,2025-03-11T11:57:05Z
1,qiskit_openqasm,1510,361,2026-07-27T12:30:55Z
2,mantidproject_mantid,240,139,2026-09-28T18:25:51Z
3,mic-dkfz_nnunet,8913,2493,2026-09-25T06:59:18Z
4,cambridgemolecularengineering_tabledataextractor,17,4,2025-03-14T20:49:17Z
5,r-spatial_gstat,215,53,2026-04-29T16:38:41Z
6,llnl_zfp,886,182,2026-09-23T00:00:44Z
7,ros-perception_slam_gmapping,736,525,2024-08-21T00:33:41Z
8,ofloveandhate_paramotopy,3,1,2025-09-30T13:56:20Z
9,qzhu2017_pyxtal,392,83,2026-09-09T16:04:04Z


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [ ]:
GitHub Repository Information

rafavzqz_geopdes
Stars: 124
Forks: 51
Last commit date: 2025-03-11

qiskit_openqasm
Stars: 1510
Forks: 361
Last commit date: 2026-07-27

mantidproject_mantid
Stars: 240
Forks: 139
Last commit date: 2026-09-28

mic-dkfz_nnunet
Stars: 8913
Forks: 2493
Last commit date: 2026-09-25

cambridgemolecularengineering_tabledataextractor
Stars: 17
Forks: 4
Last commit date: 2025-03-14

r-spatial_gstat
Stars: 215
Forks: 53
Last commit date: 2026-04-29

llnl_zfp
Stars: 886
Forks: 182
Last commit date: 2026-09-23

ros-perception_slam_gmapping
Stars: 736
Forks: 525
Last commit date: 2024-08-21

ofloveandhate_paramotopy
Stars: 3
Forks: 1
Last commit date: 2025-09-30

qzhu2017_pyxtal
Stars: 392
Forks: 83
Last commit date: 2026-09-09


In [34]:
print(woc_stats.to_string(index=False))

                                   project_wocid  number_of_commits  number_of_authors   max_time   min_time
cambridgemolecularengineering_tabledataextractor                318                 10 1741985357 1538578979
                                        llnl_zfp               2038                 84 1776082926 1458785080
                            mantidproject_mantid             176744                530 1776206119 1175694466
                                 mic-dkfz_nnunet               2575                245 1762171932 1555488318
                        ofloveandhate_paramotopy                902                  7 1759240580 1301344085
                                 qiskit_openqasm               2033                158 1761847123 1484078326
                                 qzhu2017_pyxtal               2317                 37 1762394066 1522908592
                                 r-spatial_gstat                538                 20 1774885680 1450708268
                   

The requested number of commits, the number of authors, and max and min time for each project 
based on WoC commits is listed above